In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [4]:
df=pd.read_csv("sms spam.csv")
df

,Unnamed: 0,v1,v2
0,0,ham,"Go until jurong point, crazy.. Available only ..."
1,1,ham,Ok lar... Joking wif u oni...
2,2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,3,ham,U dun say so early hor... U c already then say...
4,4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...,...
5567,5567,spam,This is the 2nd time we have tried 2 contact u...
5568,5568,ham,Will Ì_ b going to esplanade fr home?
5569,5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,5570,ham,The guy did some bitching but I acted like i'd...


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  5572 non-null   int64 
 1   v1          5572 non-null   object
 2   v2          5572 non-null   object
dtypes: int64(1), object(2)
memory usage: 130.7+ KB


In [6]:
df.describe()

,Unnamed: 0
count,5572.000000
mean,2785.500000
std,1608.642181
min,0.000000
25%,1392.750000
50%,2785.500000
75%,4178.250000
max,5571.000000


In [7]:
df.isna().sum()

Unnamed: 0    0
v1            0
v2            0
dtype: int64

In [8]:
df.duplicated().sum()

0

In [10]:
df.drop_duplicates(inplace=True)
df['v1'].value_counts()

ham     4825
spam     747
Name: v1, dtype: int64

In [12]:
df['spam']=df['v1'].apply(lambda x: 1 if x=='spam' else 0 )
df['spam']

0       0
1       0
2       1
3       0
4       0
       ..
5567    1
5568    0
5569    0
5570    0
5571    0
Name: spam, Length: 5572, dtype: int64

# Text preprocessing

we used nltk to remove stopwords and lemmatize the words

In [13]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

In [14]:
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\karne\AppData\Roaming\nltk_data...
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\karne\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\karne\AppData\Roaming\nltk_data...


True

In [15]:
lm =WordNetLemmatizer()

In [16]:
# cleaning the text
def pre_processing (word):
  word=re.sub("[^a-zA-Z]",' ',word) # remove punctuat
  word=word.lower().split() # split the sentance to be every word alone (tokenzing)
  word=[lm.lemmatize(word) for word in word if word not in stopwords.words('english')] # Lemmatization every word and remove stop words
  word=' '.join(word)
  return word

In [17]:
df['v2'][3]

'U dun say so early hor... U c already then say...'

In [18]:
# test the function
pre_processing('U dun say so early hor... U c already then say...')

'u dun say early hor u c already say'

In [19]:
# apply it on the DataFrame
df['pre_processing']=df['v2'].apply(pre_processing)
df

,Unnamed: 0,v1,v2,spam,pre_processing
0,0,ham,"Go until jurong point, crazy.. Available only ...",0,go jurong point crazy available bugis n great ...
1,1,ham,Ok lar... Joking wif u oni...,0,ok lar joking wif u oni
2,2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1,free entry wkly comp win fa cup final tkts st ...
3,3,ham,U dun say so early hor... U c already then say...,0,u dun say early hor u c already say
4,4,ham,"Nah I don't think he goes to usf, he lives aro...",0,nah think go usf life around though
...,...,...,...,...,...
5567,5567,spam,This is the 2nd time we have tried 2 contact u...,1,nd time tried contact u u pound prize claim ea...
5568,5568,ham,Will Ì_ b going to esplanade fr home?,0,b going esplanade fr home
5569,5569,ham,"Pity, * was in mood for that. So...any other s...",0,pity mood suggestion
5570,5570,ham,The guy did some bitching but I acted like i'd...,0,guy bitching acted like interested buying some...


# Modeling

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score

from sklearn.pipeline import Pipeline,make_pipeline

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.model_selection import GridSearchCV,RandomizedSearchCV

import warnings
warnings.filterwarnings('ignore')

In [21]:
x=df['pre_processing']
y=df['spam']

# Split the data

In [22]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,stratify=y,random_state=42)
x_train

184                               going nothing great bye
2171                                     wont wat wit guy
5422                           ok k sry knw siva tat askd
4113           stand away heart ache without wonder crave
4588                          finished work yet something
                              ...                        
1932                            jus finished avatar nigro
5316                             jus finish watching tv u
2308    moby pub quiz win high street prize u know new...
1903    free entry weekly comp chance win ipod txt pod...
763     nothing jus tot u would ask co u ba gua went m...
Name: pre_processing, Length: 4457, dtype: object

In [23]:
vc=TfidfVectorizer()
x_train_vec=vc.fit_transform(x_train)
x_test_vec=vc.transform(x_test)

In [24]:
x_train_vec.shape

(4457, 6255)

# Using Machine Learning

In [25]:
models=[
    ('LR',LogisticRegression()),
    ('NB',MultinomialNB()),
    ('RF',RandomForestClassifier()),
    ('Knn',KNeighborsClassifier()),
    ('SVC',SVC())
]

In [26]:
for model in models:
  print(model[0])

  model=model[1]

  model.fit(x_train_vec,y_train)

  y_pred=model.predict(x_test_vec)

  report_cls = classification_report(y_test, y_pred)

  print(report_cls)
  print('-'*30)

LR
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       966
           1       0.99      0.78      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115

------------------------------
NB
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       0.99      0.76      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115

------------------------------
RF
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       966
           1       1.00      0.82      0.90       149

    accuracy                           0.98      1115
   macro avg       0.99      0.91      0.94      1115
wei

# Trying Deep Learning Uing LSTM

In [27]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [28]:
model=keras.Sequential([
    keras.layers.LSTM(64,input_shape=(6164,1) ,return_sequences=True,activation='tanh'),

    keras.layers.LSTM(32,activation='tanh'),

    keras.layers.Dense(10,activation='relu'),
    keras.layers.Dense(1,activation='sigmoid')
])
model.compile(optimizer='adam',
                loss='binary_crossentropy',
                metrics=['accuracy']

                )
model.fit(x_train_vec.toarray(),y_train,batch_size=64,epochs=1,verbose=1,validation_data=(x_test_vec.toarray(), y_test))

70/70 ━━━━━━━━━━━━━━━━━━━━ 676s 10s/step - accuracy: 0.8615 - loss: 0.5191 - val_accuracy: 0.8664 - val_loss: 0.3946


In [29]:
model.evaluate(x_test_vec.toarray(),y_test)

35/35 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.8739 - loss: 0.3804


[0.39390864968299866, 0.8663676977157593]